# 2. Feature engineering and data preprocessing
Remove exact duplicates, retain the stratified 60/20/20 split and export reproducible inputs.

Employee ID and target are excluded from predictors. Learn imputation, encoding, feature medians and scaling from training only. Keep original predictors in additive engineering experiments; explicit removal/replacement variants are named separately. Keep plausible outliers and flagged time values pending clarification.

Numerics use median imputation, nominal categories use Missing, and ordinal categories use the training mode and stated order. LR/NN variants standardize numeric and ordinal inputs. Tree variants remain unscaled; one-hot indicators remain 0/1. Full one-hot preserves early baselines; reference encoding drops one nominal category for later experiments. No separate min-max normalization is applied.

CSV exports are for fixed-split fits and checks. **CV/search must use raw training rows with preprocessing inside each pipeline**, preventing fold leakage. Test exports belong to the frozen final assessment only.

In [1]:
# Import the tools used in this notebook.
import os
for setting in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[setting] = '1'
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
# Locate the repository even before its local data has been downloaded.
from pathlib import Path
PROJECT = Path.cwd()
if not (PROJECT / '01_EDA.ipynb').exists():
    PROJECT = PROJECT / 'attrition'
assert (PROJECT / '01_EDA.ipynb').exists(), 'Open from the repository root or its parent.'

# Skip identical CSV rewrites so a repeat run does not needlessly reopen existing files.
def save_csv(frame, path, **options):
    contents = frame.to_csv(**options).encode('utf-8')
    if not path.exists() or path.read_bytes() != contents:
        path.write_bytes(contents)


# Recreate generated folders that are intentionally absent from Git.
for folder in ['artifacts/preprocessing', 'data/preprocessed']:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)


In [2]:
# Read English data and identify exact duplicate rows.
emp = pd.read_csv(PROJECT / 'data/raw/employee_attrition_data_english.csv')
assert emp['employee_id'].notna().all() and emp['attrition'].isin([0, 1]).all()
emp_validated = emp.drop_duplicates().copy()
assert emp_validated['employee_id'].is_unique
print('Original rows:', len(emp), '| Exact duplicates removed:', len(emp) - len(emp_validated))

Original rows: 5030 | Exact duplicates removed: 30


In [3]:
# Reproduce the 60/20/20 stratified split before learning any preprocessing.
from sklearn.model_selection import train_test_split
X = emp_validated.drop(columns=['employee_id', 'attrition'])
y = emp_validated['attrition'].astype(int)
X.index.name = y.index.name = 'row_id'
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)
splits = {'train': (X_train, y_train), 'validation': (X_val, y_val), 'test': (X_test, y_test)}
assert (len(X_train), len(X_val), len(X_test)) == (3000, 1000, 1000)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
display(pd.DataFrame({
    name: {'Rows': len(target), 'Stays': int((target == 0).sum()),
           'Leavers': int(target.sum()), 'Attrition rate': target.mean()}
    for name, (_, target) in splits.items()
}).T)

,Rows,Stays,Leavers,Attrition rate
train,3000.0,2510.0,490.0,0.163333
validation,1000.0,836.0,164.0,0.164000
test,1000.0,837.0,163.0,0.163000


In [4]:
# Save raw predictors, aligned targets and separate employee membership.
split_folder = PROJECT / 'data/splits'
split_folder.mkdir(parents=True, exist_ok=True)
membership = []
for name, (predictors, target) in splits.items():
    save_csv(predictors, split_folder / f'X_{name}.csv', index_label='row_id')
    save_csv(target, split_folder / f'y_{name}.csv', index_label='row_id')
    members = emp_validated.loc[predictors.index, ['employee_id', 'attrition']].copy()
    members['split'] = name
    membership.append(members)
save_csv(pd.concat(membership), split_folder / 'employee_splits.csv', index_label='row_id')

## Reusable transformations

In [5]:
# Define column roles and ordinal category order.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
num_cols = X_train.select_dtypes(include='number').columns.tolist()
ordinal_cols = ['education', 'travel_frequency']
nominal_cols = ['gender', 'marital_status', 'department', 'job_level']
education_order = ["Associate degree", "Bachelor's degree", "Master's degree", "Doctorate"]
travel_order = ['Never', 'Occasionally', 'Frequently']

In [6]:
# Build reusable preprocessing for scaled and unscaled models.
from sklearn.pipeline import Pipeline

# Each call creates a separate set of imputers, encoders, and scalers.
def make_preprocessor(scaled=True):
    numeric_steps = [('impute', SimpleImputer(strategy='median'))]
    ordinal_steps = [
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(
            categories=[education_order, travel_order],
            handle_unknown='use_encoded_value', unknown_value=-1
        ))
    ]

    if scaled:
        numeric_steps.append(('scale', StandardScaler()))
        ordinal_steps.append(('scale', StandardScaler()))

    nominal_steps = [
        ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]

    return ColumnTransformer([
        ('nominal', Pipeline(nominal_steps), nominal_cols),
        ('ordinal', Pipeline(ordinal_steps), ordinal_cols),
        ('numeric', Pipeline(numeric_steps), num_cols)
    ], verbose_feature_names_out=False).set_output(transform='pandas')

In [7]:
# Preserve full one-hot baselines and create reference-encoded versions.
def make_reference_preprocessor(scaled=True):
    return make_preprocessor(scaled).set_params(nominal__encode__drop='first')

In [8]:
# Create features inside the pipeline so each CV fold learns its own salary medians.
from sklearn.base import BaseEstimator, TransformerMixin

class EmployeeFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, kind='none', drop_job_level=False, drop_salary=False):
        self.kind = kind
        self.drop_job_level = drop_job_level
        self.drop_salary = drop_salary

    def fit(self, X, y=None):
        if self.kind == 'relative':
            self.salary_by_level_ = X.groupby('job_level')['monthly_salary'].median()
            self.salary_fallback_ = X['monthly_salary'].median()
        return self

    def transform(self, X):
        data = X.copy()
        if self.kind == 'interactions':
            dissatisfaction = 6 - data['job_satisfaction']
            data['overtime_dissatisfaction'] = data['overtime_hours'] * dissatisfaction
            data['combined_dissatisfaction'] = dissatisfaction * (6 - data['env_satisfaction'])
        elif self.kind == 'logs':
            data['monthly_salary'] = np.log1p(data['monthly_salary'])
            data['overtime_hours'] = np.log1p(data['overtime_hours'])
        elif self.kind == 'relative':
            level_median = data['job_level'].map(self.salary_by_level_)
            level_median = level_median.fillna(self.salary_fallback_)
            data['relative_salary'] = data['monthly_salary'] / level_median

        if self.drop_job_level:
            data = data.drop(columns='job_level')
        if self.drop_salary:
            data = data.drop(columns='monthly_salary')
        return data

In [9]:
# Build a fresh pipeline for each exported experiment.
def make_export_pipeline(scaled=True, reference=True, kind='none',
                         drop_job_level=False, drop_salary=False, excluded=None):
    excluded = list(excluded or [])
    if drop_job_level:
        excluded.append('job_level')
    if drop_salary:
        excluded.append('monthly_salary')
    added = {
        'none': [], 'logs': [],
        'interactions': ['overtime_dissatisfaction', 'combined_dissatisfaction'],
        'relative': ['relative_salary'],
        'combined': ['relative_salary', 'overtime_dissatisfaction', 'combined_dissatisfaction']
    }[kind]
    preprocessor = make_reference_preprocessor(scaled) if reference else make_preprocessor(scaled)
    preprocessor.transformers = [
        (name, steps, [col for col in columns if col not in excluded]
         + (added if name == 'numeric' else []))
        for name, steps, columns in preprocessor.transformers
    ]
    orders = {'education': education_order, 'travel_frequency': travel_order}
    preprocessor.set_params(ordinal__encode__categories=[
        orders[col] for col in ordinal_cols if col not in excluded
    ])
    if kind == 'combined':
        steps = [('relative', EmployeeFeatures('relative')),
                 ('interactions', EmployeeFeatures('interactions'))]
    else:
        steps = [('features', EmployeeFeatures(kind, drop_job_level, drop_salary))]
    return Pipeline(steps + [('preprocess', preprocessor)])

## Export the variants
Each variant has train.csv, validation.csv and test.csv with predictors only. Shared targets are in data/splits/y_*.csv, aligned by row_id.

Unscaled full one-hot belongs to initial tree baselines/tuning; unscaled reference belongs to later forest importance. Relative salary uses job-level training medians with an overall training-median fallback. Interactions combine overtime/dissatisfaction and job/environment dissatisfaction. Logs replace salary/overtime; additive/combined versions keep their original inputs. Engineered/reduced versions remain experiments.

In [10]:
# Preserve both historical tree encodings and all main LR/NN feature experiments.
variants = {
    'scaled_full_onehot': {'scaled': True, 'reference': False},
    'unscaled_full_onehot': {'scaled': False, 'reference': False},
    'scaled_reference': {},
    'unscaled_reference': {'scaled': False},
    'interactions_scaled': {'kind': 'interactions'},
    'logs_scaled': {'kind': 'logs'},
    'relative_added_scaled': {'kind': 'relative'},
    'relative_no_job_scaled': {'kind': 'relative', 'drop_job_level': True},
    'relative_replace_both_scaled': {
        'kind': 'relative', 'drop_job_level': True, 'drop_salary': True},
    'combined_engineered_scaled': {'kind': 'combined'},
    'reduced_three_scaled': {'excluded': ['gender', 'work_life_balance', 'num_companies']}
}

In [11]:
# Fit each variant on training rows only, then transform/export all splits.
import cloudpickle
fitted_preprocessors = {}
catalog = []
for variant, settings in variants.items():
    pipeline = make_export_pipeline(**settings)
    pipeline.fit(X_train)
    fitted_preprocessors[variant] = pipeline
    folder = PROJECT / 'data/preprocessed' / variant
    folder.mkdir(parents=True, exist_ok=True)
    columns = None
    for name, (predictors, target) in splits.items():
        transformed = pipeline.transform(predictors)
        assert transformed.index.equals(target.index)
        assert np.isfinite(transformed.to_numpy()).all()
        assert 'attrition' not in transformed and 'employee_id' not in transformed
        if columns is None:
            columns = transformed.columns.tolist()
        assert transformed.columns.tolist() == columns
        save_csv(transformed, folder / f'{name}.csv', index_label='row_id')
    catalog.append({'Variant': variant, 'Features': len(columns),
                    'Scaled': settings.get('scaled', True),
                    'Reference encoding': settings.get('reference', True)})
with (PROJECT / 'artifacts/preprocessing/fitted_preprocessors.pkl').open('wb') as file:
    cloudpickle.dump(fitted_preprocessors, file)
preprocessing_catalog = pd.DataFrame(catalog).set_index('Variant')
save_csv(preprocessing_catalog, PROJECT / 'data/preprocessed/catalog.csv')
display(preprocessing_catalog)

,Features,Scaled,Reference encoding
Variant,,,
scaled_full_onehot,43,True,False
unscaled_full_onehot,43,False,False
scaled_reference,39,True,True
unscaled_reference,39,False,True
interactions_scaled,41,True,True
logs_scaled,39,True,True
relative_added_scaled,40,True,True
relative_no_job_scaled,35,True,True
relative_replace_both_scaled,34,True,True


## Training-only collinearity diagnostics
Individual VIF, whole-predictor group VIF and unpenalized omnibus likelihood-ratio tests answer different questions. A large VIF or small p-value alone does not justify removing a useful predictor. Notebook 3 preserves the regularized stability, forest importance and feature-removal comparisons.

In [12]:
# Use training rows only for collinearity diagnostics.
X_train_reference = fitted_preprocessors['unscaled_reference'].transform(X_train)
X_train_reference_scaled = fitted_preprocessors['scaled_reference'].transform(X_train)
encoder = fitted_preprocessors['unscaled_reference'].named_steps['preprocess'].named_transformers_['nominal'].named_steps['encode']
reference_categories = pd.Series({
    col: categories[drop_index]
    for col, categories, drop_index in zip(nominal_cols, encoder.categories_, encoder.drop_idx_)
}, name='Reference category')
display(reference_categories)

gender                      Female
marital_status            Divorced
department        Customer Service
job_level                   Expert
Name: Reference category, dtype: object

In [13]:
# Check training data for constant predictors and strongly correlated numerical pairs.
constant_features = X_train_reference.columns[X_train_reference.nunique() <= 1].tolist()
print('Constant features:', constant_features)

correlation = X_train_reference[num_cols].corr()
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
strong_pairs = upper.stack().rename('Correlation').reset_index()
strong_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
strong_pairs = strong_pairs[strong_pairs['Correlation'].abs() >= 0.80]
display(strong_pairs.round(3))


Constant features: []


,Feature 1,Feature 2,Correlation


In [14]:
# Calculate VIF using training predictors and an intercept.
# A VIF above 5 is a review flag, not an automatic reason to remove a feature.
from statsmodels.api import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_input = X_train_reference.drop(columns=constant_features)
vif_input = add_constant(vif_input.astype(float), has_constant='add')
vif_results = pd.DataFrame({
    'Feature': vif_input.columns[1:],
    'VIF': [variance_inflation_factor(vif_input.to_numpy(), i)
            for i in range(1, vif_input.shape[1])]
}).sort_values('VIF', ascending=False)
display(vif_results.round(2))
print('Features with VIF above 5:', int((vif_results['VIF'] > 5).sum()))


Features with VIF above 5: 4


,Feature,VIF
13,job_level_Junior,31.44
15,job_level_Mid-level,19.60
23,monthly_salary,19.45
17,job_level_Senior,5.69
12,department_Technology,4.62
4,marital_status_Single,4.59
2,marital_status_Married,4.07
21,total_work_years,3.95
11,department_Sales,2.52
28,job_satisfaction,2.51


Training checks: 43 encoded columns become 39 after dropping four reference categories. The design matrix including the intercept has full column rank. Maximum VIF: 31.44. No predictors have been removed based on these diagnostics.

Unknown nominal categories also map to all-zero indicators and therefore share the reference representation. For future models use `make_reference_preprocessor(...)` inside each model pipeline so imputation and encoding are fitted within each CV fold. Next: compare model-specific feature selection against these baselines using training CV and validation data.


In [15]:
# Measure collinearity for whole predictors, grouping each categorical feature's dummies.
feature_groups = {
    col: [c for c in X_train_reference.columns if c.startswith(col + '_')]
    for col in nominal_cols
}
feature_groups.update({col: [col] for col in ordinal_cols + num_cols})

# Use log determinants for a stable calculation of design-based group VIF.
correlation_matrix = X_train_reference.corr()
logdet_all = np.linalg.slogdet(correlation_matrix)[1]
group_vif_rows = []
for feature, columns in feature_groups.items():
    others = correlation_matrix.columns.difference(columns)
    log_gvif = (
        np.linalg.slogdet(correlation_matrix.loc[columns, columns])[1]
        + np.linalg.slogdet(correlation_matrix.loc[others, others])[1]
        - logdet_all
    )
    df = len(columns)
    group_vif_rows.append({
        'Feature': feature, 'df': df,
        'GVIF': np.exp(log_gvif),
        'Adjusted GVIF': np.exp(log_gvif / (2 * df))
    })

group_vif = pd.DataFrame(group_vif_rows).set_index('Feature')
group_vif = group_vif.sort_values('Adjusted GVIF', ascending=False)
display(group_vif.round(3))


,df,GVIF,Adjusted GVIF
Feature,,,
monthly_salary,1,19.451,4.410
total_work_years,1,3.947,1.987
job_satisfaction,1,2.515,1.586
num_companies,1,2.327,1.525
job_level,5,15.718,1.317
tenure,1,1.692,1.301
env_satisfaction,1,1.674,1.294
job_involvement,1,1.664,1.290
age,1,1.582,1.258


In [16]:
# Separately test the joint contribution of job level and salary on training data.
# These exploratory likelihood-ratio tests use unpenalized logistic models.
import statsmodels.api as sm
from scipy.stats import chi2

design = sm.add_constant(X_train_reference_scaled, has_constant='add')
full_logit = sm.GLM(y_train, design, family=sm.families.Binomial()).fit()
omnibus_rows = []
for name, columns in {
    'Job level': feature_groups['job_level'],
    'Monthly salary': ['monthly_salary'],
    'Job level and salary': feature_groups['job_level'] + ['monthly_salary']
}.items():
    reduced = sm.GLM(y_train, design.drop(columns=columns),
                     family=sm.families.Binomial()).fit()
    statistic = max(0, 2 * (full_logit.llf - reduced.llf))
    omnibus_rows.append({'Removed group': name, 'LR statistic': statistic,
                         'df': len(columns), 'p_value': chi2.sf(statistic, len(columns))})

omnibus_results = pd.DataFrame(omnibus_rows).set_index('Removed group')
display(omnibus_results.round(4))
print('Full unpenalized logistic model converged:', full_logit.converged)


Full unpenalized logistic model converged: True


,LR statistic,df,p_value
Removed group,,,
Job level,8.6375,5,0.1244
Monthly salary,1.2413,1,0.2652
Job level and salary,28.3306,6,0.0001


No features are automatically removed. Later engineering/removal gains were small and inconsistent. Final candidates retain all 25 original predictors with scaled reference encoding, producing 39 inputs.

In [17]:
# Record data provenance, schema, export definitions, versions and file hashes.
import hashlib, platform, sklearn, scipy
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
manifest = {
    'source_notebook': 'AAB26_HRA_Assignment_Hemanth_Kumar_Akunuri.ipynb',
    'dataset_url': 'https://www.kaggle.com/datasets/yangjiu/employee-attrition-early-warning-dataset',
    'raw_sha256': sha256(PROJECT / 'data/raw/employee_attrition_data_english.csv'),
    'seed': 42, 'target': 'attrition', 'identifier': 'employee_id',
    'raw_rows': len(emp), 'deduplicated_rows': len(emp_validated),
    'row_id': 'Original English CSV row index retained after deduplication.',
    'split_sizes': {name: len(target) for name, (_, target) in splits.items()},
    'raw_predictors': X_train.columns.tolist(),
    'raw_dtypes': {col: str(dtype) for col, dtype in X_train.dtypes.items()},
    'feature_columns': {variant: pipeline.transform(X_train.iloc[:1]).columns.tolist()
                        for variant, pipeline in fitted_preprocessors.items()},
    'nominal_columns': nominal_cols, 'ordinal_columns': ordinal_cols, 'numeric_columns': num_cols,
    'education_order': education_order, 'travel_order': travel_order,
    'variants': variants,
    'feature_counts': preprocessing_catalog['Features'].to_dict(),
    'preprocessing_fitted_on': 'training_rows_only',
    'csv_use': 'Fixed-split inputs. CV must refit preprocessing from raw training folds.',
    'test_status': 'Already evaluated historically with frozen models and validation thresholds; no retuning.',
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__,
                 'numpy': np.__version__, 'sklearn': sklearn.__version__, 'scipy': scipy.__version__,
                 'cloudpickle': cloudpickle.__version__},
    'files': {}
}
for folder in ['data/raw', 'data/splits', 'data/preprocessed', 'artifacts/preprocessing']:
    for path in sorted((PROJECT / folder).rglob('*')):
        if path.is_file():
            manifest['files'][path.relative_to(PROJECT).as_posix()] = sha256(path)
(PROJECT / 'artifacts/manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved raw splits, 11 preprocessing variants, fitted preprocessors and manifest.')

Saved raw splits, 11 preprocessing variants, fitted preprocessors and manifest.
